# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/koushalkarthik15/mlflyrankkarthik/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**Action:** `manual_review_for_refresh`
**Reason Code:** `high_decay_risk`

**The Logic:** The queue sorts pages by their modeled probability of near-term traffic decay. Pages at the very top of the list are high-traffic, older pages showing structural signs of slipping ranking. They represent the largest potential revenue loss if left untouched.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended Use:** This playbook is a directional decision-support tool for Editorial and SEO teams. They pull the Top 100 pages every week to prioritize their workflow, ensuring they are protecting high-value pages before traffic bottoms out.

**Limits:** This is purely an observed historical pattern model. It does not know *why* a page is decaying (e.g., a core algorithm update vs. competitor action), and it cannot detect semantic "evergreen" intent (e.g., a Privacy Policy that naturally ages without needing a refresh).

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Human Review:** A human editor MUST verify the search intent of the page. If the page is an archival report (e.g., "Q1 2024 Earnings"), it should be ignored.

**The No-Go List:** 
1. Do NOT automate page deletions based on this score.
2. Do NOT automate AI content rewrites directly from this queue. 
The cost of a false positive leading to a bad automated rewrite is catastrophic to domain authority.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The model should be closely monitored and retrained if:
1. **Precision Collapse:** The Precision@50 metric on the latest 30-day window drops below our naive baseline threshold.
2. **Base Rate Shift:** A Google Core Update fundamentally changes how content freshness is rewarded, drastically altering the base rate of decay across the portfolio.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

# 1. Load Data
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
features = ['content_age_days', 'impressions_90d', 'clicks_90d', 'avg_position', 'ctr', 'word_count']
df[features] = df[features].fillna(0)

# 2. Train Model on 80%, Predict on 20% Holdout
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(train[features], train['is_declining'])

# 3. Build the Playbook Queue
test['decay_probability'] = model.predict_proba(test[features])[:, 1]
playbook_queue = test.sort_values(by='decay_probability', ascending=False).copy()
playbook_queue['action'] = 'manual_review_for_refresh'
playbook_queue['reason_code'] = 'high_decay_risk'

# 4. Export
os.makedirs('../outputs', exist_ok=True)
output_path = '../outputs/action_playbook_queue.csv'
export_cols = ['content_id', 'decay_probability', 'action', 'reason_code'] + features
playbook_queue[export_cols].to_csv(output_path, index=False)
print(f"Saved Playbook Queue with {len(playbook_queue)} rows to {output_path}")

Saved Playbook Queue with 6163 rows to ../outputs/action_playbook_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.